In [ ]:
!pip install sklearn_crfsuite

In [ ]:
import csv
import random
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [ ]:
# Loading the data

train_df = pd.read_csv("C:/Users/white/Desktop/ANLP Project/SubTaskA/train.csv")

In [ ]:
train_df

1. Random baseline

In [ ]:
train_df.loc[:, "random_baseline"] = 0

In [ ]:
np.random.seed(42)
train_df["Label"] = np.random.randint(0, 2, size=len(train_df))

In [ ]:
train_df

1.1 Own testing

In [ ]:
from sklearn.metrics import f1_score

f1_score(list(train_df.random_baseline), list(train_df.Label), average = 'macro')

2. LogReg

In [ ]:
!pip install nltk

In [ ]:
import string
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
english_stopwords = stopwords.words("english")
port_stopwords = nltk.corpus.stopwords.words('portuguese')

In [ ]:
def tokenize_eng(raw_text: str):

    filtered_tokens = []

    filtered_tokens = [i.lower() for i in raw_text.split() if ( i not in string.punctuation )]
    filtered_tokens = [i for i in filtered_tokens if ( i not in english_stopwords )]
    filtered_tokens = [i for i in filtered_tokens if ( len(i) > 2 )]

    return filtered_tokens

def tokenize_port(raw_text: str):

    filtered_tokens = []

    filtered_tokens = [i.lower() for i in raw_text.split() if ( i not in string.punctuation )]
    filtered_tokens = [i for i in filtered_tokens if ( i not in port_stopwords )]
    filtered_tokens = [i for i in filtered_tokens if ( len(i) > 2 )]

    return filtered_tokens

In [ ]:
train_df_eng = train_df.iloc[:3327]    #splitting by language
train_df_port = train_df.iloc[3327:]

In [ ]:
tokenized_texts_eng= train_df_eng["sentence1"].apply(tokenize_eng)
tokenized_texts_port= train_df_port["sentence1"].apply(tokenize_port)
train_df_eng = train_df_eng.assign(tokenized=tokenized_texts_eng)
train_df_port = train_df_port.assign(tokenized=tokenized_texts_port)
train_df_eng.tokenized = train_df_eng.tokenized.apply(lambda code: " ".join(code))
train_df_port.tokenized = train_df_port.tokenized.apply(lambda code: " ".join(code))

In [ ]:
!pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
!python -m spacy download pt_core_news_sm

In [ ]:
import spacy
from spacy.lang.pt.examples import sentences 
from spacy.lang.en.examples import sentences 

en = spacy.load("en_core_web_sm")
pt = spacy.load("pt_core_news_sm")

def lemmatize_eng(raw_text: str):

    lemmatized = []
    doc = en(raw_text)
    lemmatized = [w.lemma_ for w in doc]
    lemmatized = [word.strip(string.punctuation) for word in lemmatized]
    return lemmatized

def lemmatize_port(raw_text: str):

    lemmatized = []
    doc = pt(raw_text)
    lemmatized = [w.lemma_ for w in doc]
    lemmatized = [word.strip(string.punctuation) for word in lemmatized]
    return lemmatized

In [ ]:
vects1_eng = train_df_eng["tokenized"].apply(lemmatize_eng)
train_df_eng = train_df_eng.assign(lemmatized=vects1_eng)
train_df_eng.lemmatized = train_df_eng.lemmatized.apply(lambda code: " ".join(code))

In [ ]:
vects1_port = train_df_port["tokenized"].apply(lemmatize_port)
train_df_port = train_df_port.assign(lemmatized=vects1_port)
train_df_port.lemmatized = train_df_port.lemmatized.apply(lambda code: " ".join(code))

In [ ]:
train_df = pd.concat([train_df_eng, train_df_port], ignore_index=True)

In [ ]:
train_df

In [ ]:
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.metrics import classification_report

In [106]:
dev_df = pd.read_csv("C:/Users/white/Desktop/ANLP Project/SubTaskA/Data/dev.csv")
dev_gold = pd.read_csv("C:/Users/white/Desktop/ANLP Project/SubTaskA/Data/dev_gold.csv")

In [101]:
len(dev_df)

739

In [107]:
dev_df['text'] = dev_df['Previous'] + dev_df['Target'] + dev_df['Next']

In [108]:
dev_df_eng = dev_df.iloc[:465]
dev_df_port = dev_df.iloc[465:]

In [109]:
tokenized_texts_eng= dev_df_eng["text"].apply(tokenize_eng)
tokenized_texts_port= dev_df_port["text"].apply(tokenize_port)
dev_df_eng = dev_df_eng.assign(tokenized=tokenized_texts_eng)
dev_df_port = dev_df_port.assign(tokenized=tokenized_texts_port)
dev_df_eng.tokenized = dev_df_eng.tokenized.apply(lambda code: " ".join(code))
dev_df_port.tokenized = dev_df_port.tokenized.apply(lambda code: " ".join(code))

In [110]:
vects1 = dev_df_eng["tokenized"].apply(lemmatize_eng)
dev_df_eng = dev_df_eng.assign(lemmatized=vects1)
dev_df_eng.lemmatized = dev_df_eng.lemmatized.apply(lambda code: " ".join(code))

In [111]:
vects1 = dev_df_port["tokenized"].apply(lemmatize_port)
dev_df_port = dev_df_port.assign(lemmatized=vects1)
dev_df_port.lemmatized = dev_df_port.lemmatized.apply(lambda code: " ".join(code))

In [112]:
dev_df = pd.concat([dev_df_eng, dev_df_port], ignore_index=True)

In [114]:
len(dev_gold)

739

In [115]:
X_train = train_df['lemmatized'].values
y_train = train_df['label'] 

In [116]:
vec = TfidfVectorizer()
X_test = dev_df['lemmatized'].values
X_train = vec.fit_transform(X_train)
X_test = vec.transform(X_test)

In [117]:
logreg = LogisticRegression()
clf = logreg.fit(X_train, y_train)

In [118]:
pred = clf.predict(X_test)

In [119]:
print(classification_report(dev_gold['Label'], pred))

              precision    recall  f1-score   support

           0       0.53      0.91      0.67       336
           1       0.81      0.33      0.47       403

    accuracy                           0.59       739
   macro avg       0.67      0.62      0.57       739
weighted avg       0.68      0.59      0.56       739



2. Naive Bayes

In [121]:
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()

X_train_dense = X_train.toarray() if hasattr(X_train, 'toarray') else np.asarray(X_train)
y_train_dense = np.asarray(y_train)

gnb.fit(X_train_dense, y_train_dense)
X_test_dense = X_test.toarray() if hasattr(X_test, 'toarray') else np.asarray(X_test)
pred = gnb.predict(X_test_dense)

print(classification_report(dev_gold['Label'], pred))

              precision    recall  f1-score   support

           0       0.48      0.70      0.57       336
           1       0.60      0.37      0.46       403

    accuracy                           0.52       739
   macro avg       0.54      0.53      0.51       739
weighted avg       0.54      0.52      0.51       739



Some comments:

1) One could look for a lemmatizer for Portugal, because now Portugal texts do not get lemmatized at all

2) For the report (Feb): maybe analyze the percentage of correct answers for ENG and PT separately; other metrics like accuracy may be worth reporting

3) Think of how we can include the left and right context